In [ ]:
import os
import numpy as np
import nibabel as nib

# Chemin réel vers les données segmentées (masquées)
segmented_data_dir = '/home/amenacer/Stage/Data/TollSome Segmentation /Segmentation/resultas/tollsome 2D+T/maskedimg'

# Dossier pour sauvegarder les résultats
features_output_dir = '/home/amenacer/Stage/Data/TollSome Segmentation /Segmentation/resultas/tollsome 2D+T/features_masked'
os.makedirs(features_output_dir, exist_ok=True)

# Parcours réel des groupes (Tollsome_output1, 2, 3)
for group_name in ['Tollsome_output1', 'Tollsome_output2', 'Tollsome_output3']:
    group_path = os.path.join(segmented_data_dir, group_name)
    group_features_dir = os.path.join(features_output_dir, group_name)
    os.makedirs(group_features_dir, exist_ok=True)

    for nii_file in os.listdir(group_path):
        if nii_file.endswith('.nii.gz'):
            nii_path = os.path.join(group_path, nii_file)

            # Charger l'image segmentée
            nii_img = nib.load(nii_path)
            nii_data = nii_img.get_fdata()

            # Calcul des aires à chaque instant temporel
            areas = [np.sum(nii_data[:, :, t] > 0) for t in range(nii_data.shape[-1])]
            areas = np.array(areas)

            # Calcul des descripteurs
            min_area = np.min(areas)
            max_area = np.max(areas)
            mean_area = np.mean(areas)
            std_area = np.std(areas)

            ascending_slope = np.max(np.diff(areas)) if areas.size > 1 else 0
            descending_slope = np.min(np.diff(areas)) if areas.size > 1 else 0

            features = np.array([
                min_area, max_area, mean_area, std_area, ascending_slope, descending_slope
            ])

            # Sauvegarde structurée des descripteurs
            feature_filename = nii_file.replace('.nii.gz', '_features.txt')
            feature_filepath = os.path.join(group_features_dir, feature_filename)
            np.savetxt(feature_filepath, features,
                       header='min_area max_area mean_area std_area ascending_slope descending_slope',
                       fmt='%.4f')

            print(f'Descripteurs sauvegardés : {feature_filepath}')

print("✅ Extraction terminée avec succès !")


In [ ]:
import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

# Chemin réel vers les données segmentées (masquées)
segmented_data_dir = '/home/amenacer/Stage/Data/TollSome Segmentation /Segmentation/resultas/tollsome 2D+T/maskedimg'

# Parcours chaque groupe et chaque image clairement
for group_name in ['Tollsome_output1', 'Tollsome_output2', 'Tollsome_output3']:
    group_path = os.path.join(segmented_data_dir, group_name)
    nii_files = [f for f in os.listdir(group_path) if f.endswith('.nii.gz')]
    
    # Nombre total de fichiers pour ajuster la taille des graphiques
    num_files = len(nii_files)
    plt.figure(figsize=(5*num_files, 4))

    for idx, nii_file in enumerate(nii_files, 1):
        nii_path = os.path.join(group_path, nii_file)

        # Charger l'image segmentée
        nii_img = nib.load(nii_path)
        nii_data = nii_img.get_fdata()

        # Calcul des aires à chaque instant temporel
        areas = [np.sum(nii_data[:, :, t] > 0) for t in range(nii_data.shape[-1])]

        # Tracer la courbe Aire vs Temps
        plt.subplot(1, num_files, idx)
        plt.plot(areas, marker='o')
        plt.title(f'{nii_file}', fontsize=10)
        plt.xlabel('Temps')
        plt.ylabel('Aire segmentée')
        plt.grid(True)

    plt.suptitle(f'Courbes Aire segmentée vs Temps ({group_name})', fontsize=15)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()
